# Codecs: how values are stored

Each field of a record is stored by a **codec**, chosen by the type of the value:

| value | codec | stored as | `record["x"]` (lazy) | `record.load("x")` |
|---|---|---|---|---|
| `numpy.ndarray` | `ndarray` | zarr v3 array in `data.zarr/<name>` | `zarr.Array` | `ndarray` |
| `xarray.DataArray`, `Dataset`, `DataTree` | `xarray` | zarr group `data.zarr/<name>` | opened without loading | loaded |
| `pandas.DataFrame` | `parquet` | `<name>.parquet` | `DataFrame` | `DataFrame` |
| `pyarrow.Table`, `RecordBatchReader` | `arrow` | `<name>.parquet`, streamed | `pyarrow.dataset.Dataset` | `pyarrow.Table` |
| JSON scalars, lists, str-keyed dicts | `json` | `<name>.json` | value | value |
| `torch.nn.Module`, `{str: Tensor}` | `safetensors` | `<name>.safetensors` | tensor dict | tensor dict |
| anything, opt-in | `pickle` | `<name>.pkl` | value | value |
| `numpy.ndarray`, opt-in | `hdf5` | `<name>.h5` | `h5py.Dataset` | `ndarray` |

This notebook stores one value of each kind, appends rows to arrays in zarr and HDF5, names
array axes, and registers a codec for a type of its own.

In [ ]:
import dataclasses
import json
import tempfile
from pathlib import Path
from typing import Annotated

import numpy as np
import pandas as pd
import pyarrow as pa
import xarray as xr
from jaxtyping import Float

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-codecs-"))
td.set_root(root)

## One field per codec

Inside `__init__` (the build), `self.name = value` and `self["name"] = value` choose the codec from the value;
`self.write(name, value, codec=...)` names it explicitly. The pickle codec is never chosen by
type: pickles run code when loaded and break when classes move.

In [ ]:
class Recording(td.Record):
    def __init__(self, /, *, n_cells: int = 4, n_frames: int = 50):
        rng = np.random.default_rng(0)
        time = np.arange(n_frames) / 10
        traces = rng.normal(size=(n_cells, n_frames)).cumsum(1)
        cell_types = ["T4a", "T4b", "T5a", "T5b"][: n_cells]

        self.raw = traces  # ndarray -> zarr array
        self.responses = xr.DataArray(  # xarray -> zarr group with coordinates
            traces, dims=("cell", "time"), coords={"cell_type": ("cell", cell_types), "time": time}, name="voltage"
        )
        self.cells = pd.DataFrame({"cell_type": cell_types, "depth_um": rng.uniform(0, 100, n_cells)})  # parquet
        self.events = pa.table({"cell": rng.integers(0, n_cells, 200), "frame": rng.integers(0, n_frames, 200)})  # arrow
        self.settings = {"sampling_hz": 10, "cell_types": cell_types}  # json
        self.write("rng_state", rng.bit_generator.state, codec="pickle")  # opt-in pickle


rec = Recording()
rec

The manifest in `_meta.json` names each field's codec and, for arrays, dtype, shape and
dimension names.

In [ ]:
print(json.dumps(rec.meta["fields"], indent=1))

Lazy and eager reads per codec: `record["x"]` opens without loading where the codec allows it,
`record.load("x")` reads into memory, and `record.x` returns the declared type of typed fields
and the lazy handle otherwise.

In [ ]:
for name in rec.keys():
    print(f"{name:<10} record[name]: {type(rec[name]).__module__}.{type(rec[name]).__name__:<12} load: {type(rec.load(name)).__name__}")

A lazy xarray field keeps its coordinates, and a lazy arrow field is a `pyarrow.dataset.Dataset`
that reads only the columns and rows a query needs.

In [ ]:
import pyarrow.dataset as ds

print(rec["responses"].sel(time=slice(0, 0.3)).cell_type.values)
early = rec["events"].to_table(columns=["cell"], filter=ds.field("frame") < 5)
print(early.num_rows, "events in the first five frames")

`RecordBatchReader`s are streamed batch by batch, so a table larger than memory can be written
from a query result (DuckDB, a Delta table, a parquet scan).

In [ ]:
class Stream(td.Record):
    def __init__(self):
        batches = (pa.record_batch({"i": np.arange(k * 1000, (k + 1) * 1000)}) for k in range(5))
        self.rows = pa.RecordBatchReader.from_batches(pa.schema({"i": pa.int64()}), batches)


stream = Stream()
print(stream.meta["fields"]["rows"], stream["rows"].count_rows())

## xarray Datasets and DataTrees

A whole `Dataset` or a `DataTree` is one field; each tree node becomes a zarr group.

In [ ]:
class Simulation(td.Record):
    def __init__(self):
        t = np.linspace(0, 1, 100)
        stimulus = xr.Dataset({"luminance": ("time", np.sin(2 * np.pi * 3 * t))}, coords={"time": t})
        layers = {
            f"/layer{i}": xr.Dataset({"rate": (("time", "unit"), np.outer(np.cos(t * i), np.ones(i + 1)))}, coords={"time": t})
            for i in range(1, 3)
        }
        self.stimulus = stimulus
        self.network = xr.DataTree.from_dict(layers)


sim = Simulation()
print(sim["stimulus"])
sim.load("network")

## Append rows

`append(name, rows)` grows an array along its first axis and creates it on first use. Rows must
match the trailing shape and are cast to the stored dtype. Training loops use it for losses
and checkpoint indices; with `in_place = True`, readers see each row while the build runs.

In [ ]:
class Training(td.Record):
    loss: Float[np.ndarray, "n_steps"]
    weights: Float[np.ndarray, "n_steps n_units"]

    def __init__(self, /, *, n_steps: int = 30):
        rng = np.random.default_rng(0)
        w = rng.normal(size=8)
        for step in range(n_steps):
            w = 0.9 * w
            self.append("loss", [float((w**2).sum())])
            self.append("weights", w[None])  # one row of shape (1, 8)


training = Training()
training.tree()
print(training.loss[:5].round(3))

A field written with `write` cannot be appended to, and appended rows must match the
trailing shape.

In [ ]:
class BadAppend(td.Record):
    def __init__(self):
        self.append("weights", np.zeros((1, 8)))
        self.append("weights", np.zeros((1, 3)))


try:
    BadAppend()
except td.BuildFailed as exc:
    print(type(exc).__name__, "->", exc.error["message"])

## HDF5 output

For tools that read `.h5` files, store arrays as HDF5: per field with `codec="hdf5"`, or for a
whole record type with `codecs = {"ndarray": "hdf5"}`, which also applies to `append`. Each
field is `<name>.h5` with one dataset `data` (datamate's layout) labeled with its dimension
names. Zarr stays the default because HDF5 relies on file locks that misbehave on NFS and
Lustre, allows a single writer per file, and cannot write chunks in parallel.

In [ ]:
import h5py


class Hdf5Training(td.Record):
    codecs = {"ndarray": "hdf5"}  # every array of this type, appended or written

    loss: Float[np.ndarray, "n_steps"]
    final: Float[np.ndarray, "n_units"]

    def __init__(self, /, *, n_steps: int = 20):
        w = np.ones(4)
        for _ in range(n_steps):
            w = 0.8 * w
            self.append("loss", [float((w**2).sum())])  # chunked dataset, resized along axis 0
        self.final = w
        self.write("as_zarr", w, codec="ndarray")  # an explicit codec overrides `codecs`


h5rec = Hdf5Training()
h5rec.tree()

with h5py.File(h5rec.path / "loss.h5", "r", locking=False) as f:  # any h5 tool can read it
    data = f["data"]
    print(data.shape, data.maxshape, data.chunks, "label:", data.dims[0].label)

A lazy HDF5 read returns an `h5py.Dataset`, whose file stays open until the handle is dropped;
`load` and typed numpy fields read into memory and close the file. A single field can opt in
without changing the record type.

In [ ]:
handle = h5rec["loss"]
print(type(handle).__name__, handle[:3].round(3))
del handle
print(type(h5rec.loss).__name__, type(h5rec.load("final")).__name__)


class Mixed(td.Record):
    def __init__(self):
        self.write("export", np.eye(3), codec="hdf5", dims=("row", "col"))
        self.append("log", [1.0, 2.0], codec="hdf5")
        self.append("log", [3.0])


Mixed().tree()

## Dimension names

jaxtyping shape strings name the axes of typed fields:

- `n_<things>` becomes `<thing>` (drop `n_` and one trailing `s`);
- irregular plurals come from a built-in table, `td.register_dims` (process-wide) or a record's
  `dims` mapping (one type);
- `name=size` tokens use `name`, so a square matrix gets distinct axes;
- `Annotated[..., td.Dims(...)]` names axes the shape string cannot (`n_bins+1`); untyped arrays
  have no dimension names unless `write` gets `dims=`.

At the end of a build, every field sharing a dimension name must agree on its size.

In [ ]:
td.register_dims(n_ommatidia="ommatidium")


class Retina(td.Record):
    dims = {"n_frames": "time"}  # this record type only

    photoreceptors: Float[np.ndarray, "n_frames n_ommatidia"]
    frame_edges: Annotated[Float[np.ndarray, "n_frames+1=_"], td.Dims("time_edge")]  # "=_" skips the size check
    coupling: Float[np.ndarray, "pre=n_ommatidia post=n_ommatidia"]
    outline: Float[np.ndarray, "n_vertices n_coords"]

    def __init__(self, /, *, n_ommatidia: int = 7, n_frames: int = 10, n_vertices: int = 6):
        self.photoreceptors = np.zeros((n_frames, n_ommatidia))
        self.frame_edges = np.linspace(0, 1, n_frames + 1)
        self.coupling = np.eye(n_ommatidia)
        self.outline = np.zeros((n_vertices, 2))


retina = Retina()
retina.tree()
print(td.dims_for(Float[np.ndarray, "n_radii n_cells"]))

The zarr store of a record opens as one labeled `xarray.Dataset`.

In [ ]:
xr.open_zarr(retina.path / "data.zarr", consolidated=False)

In [ ]:
class Inconsistent(td.Record):
    a: Float[np.ndarray, "n_points"]
    b: Float[np.ndarray, "n_points"]

    def __init__(self):
        self.a, self.b = np.zeros(3), np.zeros(4)


try:
    Inconsistent()
except td.BuildFailed as exc:
    print(exc.error["type"], "->", exc.error["message"])

## A codec for your own type

A codec claims values in `accepts`, writes them under the record directory in `write` and
returns JSON-able info for the manifest, and reads them back in `read`. `FileCodec` stores one
file `<name><suffix>`. Codecs registered later are tried first. Register a codec in the module
that defines the record class, so every process that reads the record knows it. See
[Extending](../../extending/#codecs) for torch modules, images, videos and quantized xarray.

In [ ]:
from tensordir.codecs import FileCodec


@dataclasses.dataclass
class Polygon:
    vertices: np.ndarray  # (n_vertices, 2)
    label: str


class PolygonCodec(FileCodec):
    name = "polygon"
    suffix = ".npz"

    def accepts(self, value):
        return isinstance(value, Polygon)

    def write(self, directory, name, value, *, dims=None):
        np.savez(self.path(directory, name), vertices=value.vertices)
        return {"label": value.label, "n_vertices": len(value.vertices)}

    def read(self, directory, name, info, *, lazy=True):
        with np.load(self.path(directory, name)) as data:
            return Polygon(data["vertices"], info["label"])


td.register_codec(PolygonCodec(), replace=True)  # replace=True lets this cell run twice


class Region(td.Record):
    def __init__(self):
        angles = np.linspace(0, 2 * np.pi, 6, endpoint=False)
        self.hexagon = Polygon(np.stack([np.cos(angles), np.sin(angles)], 1), "column")


region = Region()
print(region.hexagon.label, region.hexagon.vertices.shape, region.meta["fields"]["hexagon"])